## Part 3

### Task 12

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

Q = np.array([
    [-0.0085, 0.005, 0.0025, 0, 0.001],
    [0, -0.014, 0.005, 0.004, 0.005],
    [0, 0, -0.008, 0.003, 0.005],
    [0, 0, 0, -0.009, 0.009],
    [0, 0, 0, 0, 0]
])

In [ ]:
def simulate_ctmc(Q, start_state, max_time):
    state = start_state
    time_now = 0
    states = [state]
    times = [0]

    while state != 4 and time_now < max_time:
        rate = -Q[state, state]
        sojourn = np.random.exponential(1 / rate)
        time_now = time_now + sojourn

        if time_now >= max_time:
            break

        probs = Q[state].copy()
        probs[state] = 0
        probs = probs / rate
        state = np.random.choice(5, p=probs)

        states.append(state)
        times.append(time_now)

    return times, states

In [ ]:
def state_at_time(times, states, t):
    state = states[0]

    for i in range(len(times)):
        if times[i] <= t:
            state = states[i]
        else:
            break

    return state

In [ ]:
n = 1000
step = 48
max_steps = 40

all_times = []
all_states = []

for i in range(n):
    times, states = simulate_ctmc(Q, 0, 100000)
    all_times.append(times)
    all_states.append(states)

In [ ]:
observations = []

for i in range(n):
    times = all_times[i]
    states = all_states[i]
    obs = []
    t = 0

    for step_number in range(max_steps):
        s = state_at_time(times, states, t)
        obs.append(s)

        if s == 4:
            break

        t = t + step

    observations.append(obs)

lengths = [len(obs) for obs in observations]
last_states = [obs[-1] for obs in observations]

print("Shortest observed time series has", min(lengths), "entries")
print("Longest observed time series has", max(lengths), "entries")
print("Number of women whose series ends in the death state:", last_states.count(4))
print("Example time series for woman 1:", observations[0])
print("Example time series for woman 2:", observations[1])

### Task 13

In [ ]:
def simulate_bridge(Q, start_state, end_state, duration, max_tries):
    for attempt in range(max_tries):
        times, states = simulate_ctmc(Q, start_state, duration)
        final_state = state_at_time(times, states, duration)

        if final_state == end_state:
            jump_times = []
            jump_states = []

            for i in range(len(times)):
                if times[i] < duration:
                    jump_times.append(times[i])
                    jump_states.append(states[i])

            return jump_times, jump_states, True

    return None, None, False

In [ ]:
def reconstruct_trajectories(Q, observations, step, max_tries):
    N = np.zeros((5, 5))
    S = np.zeros(5)
    failed = 0

    for obs in observations:
        full_times = [0]
        full_states = [obs[0]]

        for k in range(len(obs) - 1):
            start_state = obs[k]
            end_state = obs[k + 1]
            interval_start = k * step

            jump_times, jump_states, success = simulate_bridge(Q, start_state, end_state, step, max_tries)

            if not success:
                failed = failed + 1
                continue

            for j in range(1, len(jump_times)):
                full_times.append(jump_times[j] + interval_start)
                full_states.append(jump_states[j])

            full_times.append(interval_start + step)
            full_states.append(end_state)

        for k in range(len(full_states) - 1):
            i = full_states[k]
            j = full_states[k + 1]

            if i != j:
                N[i, j] = N[i, j] + 1

            S[i] = S[i] + (full_times[k + 1] - full_times[k])

    return N, S, failed

In [ ]:
def update_Q(N, S):
    Q_new = np.zeros((5, 5))

    for i in range(4):
        for j in range(5):
            if i != j:
                Q_new[i, j] = N[i, j] / S[i]

        Q_new[i, i] = -np.sum(Q_new[i])

    return Q_new

In [ ]:
Q_estimate = np.array([
    [-0.02, 0.01, 0.005, 0, 0.005],
    [0, -0.03, 0.01, 0.01, 0.01],
    [0, 0, -0.02, 0.01, 0.01],
    [0, 0, 0, -0.02, 0.02],
    [0, 0, 0, 0, 0]
])

max_iterations = 20
tolerance = 0.001
differences = []
failed_counts = []

for iteration in range(max_iterations):
    N, S, failed = reconstruct_trajectories(Q_estimate, observations, step, max_tries=5000)
    Q_new = update_Q(N, S)
    difference = np.max(np.abs(Q_new - Q_estimate))
    differences.append(difference)
    failed_counts.append(failed)
    Q_estimate = Q_new

    if difference < tolerance:
        break

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(differences) + 1), differences, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Max absolute change in Q")
plt.title("Convergence of the MCEM algorithm")
plt.yscale("log")
plt.show()

print("Number of iterations until convergence:", len(differences))
print("Number of failed bridge reconstructions per iteration:", failed_counts)
print()
print("Estimated Q:")
print(np.round(Q_estimate, 4))
print()
print("True Q:")
print(Q)
print()
print("Max absolute error:", np.max(np.abs(Q_estimate - Q)))